# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa — Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Loaded dataset @id: {metadata['@id']}")
print(f"Name: {metadata['name']}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields using their @id
schema = dataset.schema
record_sets = schema.get('recordSet', [])
if not record_sets:
    print('No record sets defined in the root schema. Inspecting distribution-defined record sets...')
    # Often, record sets are represented as DataFileObjects and linked via distribution
    record_sets = []
    dists = schema.get('distribution', [])
    for dist in dists:
        if isinstance(dist, dict) and '@id' in dist:
            record_sets.append(dist['@id'])
    print(f"Record set (from distribution): {record_sets}")
else:
    print(f"Record sets by @id: {record_sets}")

# For detailed info, list record sets with their available fields (using @id)
from pprint import pprint
record_set_ids = []
if record_sets:
    for rs in record_sets:
        print(f'RecordSet @id: {rs}')
        record_set_ids.append(rs)
else:
    # Try to extract from distribution objects (assuming CSVs; mlcroissant will read them)
    distributions = schema.get('distribution', [])
    for dist in distributions:
        # Each DataFileObject
        if isinstance(dist, dict) and '@id' in dist:
            print(f"Distribution @id (as RecordSet): {dist['@id']}")
            record_set_ids.append(dist['@id'])

# Try to print the fields of the first recordset
if record_set_ids:
    try:
        dset = dataset.schema
        # Look for fields in the relevant distribution
        first_rs_id = record_set_ids[0]
        for dist in dset.get('distribution', []):
            if isinstance(dist, dict) and dist.get('@id') == first_rs_id:
                columns = dist.get('column', [])
                print(f'Fields for record set {first_rs_id}')
                field_ids = []
                for col in columns:
                    if isinstance(col, dict):
                        field_ids.append(col.get('@id'))
                        print(f"  Column @id: {col.get('@id')}, name: {col.get('name')}")
                    else:
                        print(f"  Column: {col}")
                break
    except Exception as e:
        print(f"Could not extract fields: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract data using the available distribution @id (record set ID) identified above
import warnings
warnings.filterwarnings('ignore')

# Use the first found record set as the example (it's a DataFileObject, e.g. a CSV)
record_sets = record_set_ids  # from the previous cell
dataframes = {}

for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set] = df
            print(f"Loaded {len(df)} records for record set: {record_set}")
        else:
            print(f"No data for record set: {record_set}")
    except Exception as e:
        print(f"Could not load records for {record_set}: {e}")

# Pick the first loaded dataframe
main_record_set_id = next(iter(dataframes))
print(f"\nMain DataFrame columns for RecordSet @id={main_record_set_id}:")
print(list(dataframes[main_record_set_id].columns))
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a numeric column such as 'gad7_score' if it exists
df = dataframes[main_record_set_id]

numeric_field = None
# Look for common mental health assessment columns
for possible_col in [
        'gad7_score', 'phq9_score', 'psq_score', 
        'GAD7_TOTAL', 'PHQ9_TOTAL', 'PSQ_TOTAL', 'score', 'total_score'
    ]:
    if possible_col in df.columns:
        numeric_field = possible_col
        break

if numeric_field is None:
    # Fallback: pick first numeric column
    for col in df.select_dtypes('number').columns:
        numeric_field = col
        break

if numeric_field:
    print(f"Selected numeric field for EDA: {numeric_field}")
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Optional grouping by a demographic, e.g. 'gender', 'village', or 'level_of_education'
    for group_field in ['gender', 'village', 'level_of_education', 'marital_status', 'Gender', 'Village', 'Level of Education']:
        if group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} grouped by {group_field} (top entries):")
            print(grouped_df.head())
            break
else:
    print('No numeric field detected in main record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot distribution of the selected numeric field
if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by a categorical variable (demographic)
    for group_field in ['gender', 'village', 'level_of_education', 'marital_status', 'Gender', 'Village', 'Level of Education']:
        if group_field in df.columns:
            plt.figure(figsize=(10,4))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
            break

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load, inspect, and perform simple EDA and visualizations on a FAIR² mental health survey dataset from Kilifi County, Kenya, using the `mlcroissant` library. For advanced analyses, continue to reference field and record set `@id`s as per the Croissant schema and apply further domain-specific processing as required.